>## Data Ingestion -> Chunking -> Embedding -> Vector db 

In [38]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [39]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Search known project locations instead of relying on the notebook's working directory.
known_locations = [
    Path(r"E:\Learning\Generative Ai\Rag\data\pdf"),
    Path.cwd() / "data" / "pdf",
    Path.cwd() / "Rag" / "data" / "pdf",
]

pdf_directory = next((path for path in known_locations if path.is_dir()), None)
print(f"Notebook/kernel working directory: {Path.cwd()}")
print(f"PDF directory resolved to: {pdf_directory}")

if pdf_directory is None:
    raise FileNotFoundError("Could not find the RAG PDF folder.")

all_pdf_documents = process_all_pdfs(pdf_directory)

Notebook/kernel working directory: e:\Learning\Generative Ai\Rag
PDF directory resolved to: E:\Learning\Generative Ai\Rag\data\pdf
Found 3 PDF files to process

Processing: chunking_20.pdf
  ✓ Loaded 20 pages

Processing: loaders_15.pdf
  ✓ Loaded 15 pages

Processing: rag_10.pdf
  ✓ Loaded 10 pages

Total documents loaded: 45


In [40]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-18T18:09:27+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-18T18:09:27+05:00', 'subject': '(unspecified)', 'title': 'Text Chunking, Embeddings, and Vector Retrieval', 'trapped': '/False', 'source': 'E:\\Learning\\Generative Ai\\Rag\\data\\pdf\\chunking_20.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1', 'source_file': 'chunking_20.pdf', 'file_type': 'pdf'}, page_content='Text Chunking, Embeddings, and Vector Retrieval\n Learning guide created for RAG document loader and retrieval practice.\nPage 1: Why documents need splitting\nLarge documents cannot always be passed to retrieval or embedding systems as one unit. Text\nsplitting divides documents into smaller chunks with enough meaning to answer a query. Chunk\nsize should balance precision and context. Small chunks can identify exact facts, but they may omit\nexplanations. Large chunk

In [41]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [42]:
chunks=split_documents(all_pdf_documents)
chunks

Split 45 documents into 90 chunks

Example chunk:
Content: Text Chunking, Embeddings, and Vector Retrieval
 Learning guide created for RAG document loader and retrieval practice.
Page 1: Why documents need splitting
Large documents cannot always be passed to ...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-18T18:09:27+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-18T18:09:27+05:00', 'subject': '(unspecified)', 'title': 'Text Chunking, Embeddings, and Vector Retrieval', 'trapped': '/False', 'source': 'E:\\Learning\\Generative Ai\\Rag\\data\\pdf\\chunking_20.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1', 'source_file': 'chunking_20.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-18T18:09:27+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-18T18:09:27+05:00', 'subject': '(unspecified)', 'title': 'Text Chunking, Embeddings, and Vector Retrieval', 'trapped': '/False', 'source': 'E:\\Learning\\Generative Ai\\Rag\\data\\pdf\\chunking_20.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1', 'source_file': 'chunking_20.pdf', 'file_type': 'pdf'}, page_content='Text Chunking, Embeddings, and Vector Retrieval\n Learning guide created for RAG document loader and retrieval practice.\nPage 1: Why documents need splitting\nLarge documents cannot always be passed to retrieval or embedding systems as one unit. Text\nsplitting divides documents into smaller chunks with enough meaning to answer a query. Chunk\nsize should balance precision and context. Small chunks can identify exact facts, but they may omit\nexplanations. Large chunk

<br><br>

<br><br>

>###  Embedding

In [43]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [44]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4088.17it/s]


Model loaded successfully. Embedding dimension: 384


<br>

>### VectorStore

In [45]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = None):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        if persist_directory is None:
            rag_dir = Path.cwd()
            if rag_dir.name.lower() != "rag":
                rag_dir = rag_dir / "Rag"
            persist_directory = str(rag_dir / "data" / "vector_store")
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 90


In [46]:
# Generate embeddings and add all 90 chunks to Chroma.
chunk_texts = [chunk.page_content for chunk in chunks]
chunk_embeddings = embedding_manager.generate_embeddings(chunk_texts)

vectorstore.add_documents(chunks, chunk_embeddings)


Generating embeddings for 90 texts...


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.19it/s]

Generated embeddings with shape: (90, 384)
Adding 90 documents to vector store...
Successfully added 90 documents to vector store
Total documents in collection: 180


In [47]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-18T18:09:27+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-18T18:09:27+05:00', 'subject': '(unspecified)', 'title': 'Text Chunking, Embeddings, and Vector Retrieval', 'trapped': '/False', 'source': 'E:\\Learning\\Generative Ai\\Rag\\data\\pdf\\chunking_20.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1', 'source_file': 'chunking_20.pdf', 'file_type': 'pdf'}, page_content='Text Chunking, Embeddings, and Vector Retrieval\n Learning guide created for RAG document loader and retrieval practice.\nPage 1: Why documents need splitting\nLarge documents cannot always be passed to retrieval or embedding systems as one unit. Text\nsplitting divides documents into smaller chunks with enough meaning to answer a query. Chunk\nsize should balance precision and context. Small chunks can identify exact facts, but they may omit\nexplanations. Large chunk

<br>

>### Rag Retriever

In [48]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                if not retrieved_docs:
                    top_scores = [round(1 - distance, 3) for distance in distances]
                    print(f"Nearest cosine similarities: {top_scores}")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [49]:
rag_retriever

In [50]:
rag_retriever.retrieve("What is the purpose of TextLoader?", top_k=3, score_threshold=0.5)

Retrieving documents for query: 'What is the purpose of TextLoader?'
Top K: 3, Score threshold: 0.5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 218.25it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Nearest cosine similarities: [0.452, 0.452, 0.447]


[]

>## Retrieval Pipeline

In [51]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage


In [52]:
import os
from dotenv import load_dotenv
load_dotenv()

class GeminiLLM:
    def __init__(self, model_name: str = "gemini-2.5-flash", api_key: str = None):
        self.model_name = model_name
        self.api_key = api_key or os.getenv("GOOGLE_API_KEY")
        if not self.api_key:
            raise ValueError("Set GOOGLE_API_KEY in E:\Learning\Generative Ai\.env")

        self.llm = ChatGoogleGenerativeAI(
            model=self.model_name,
            google_api_key=self.api_key,
            temperature=0.1,
        )
        print(f"Initialized Gemini model: {self.model_name}")

    def generate_response(self, query: str, context: str) -> str:
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""Use only the following context to answer the question accurately.\n\nContext:\n{context}
\nQuestion: {question}\n\nIf the context is insufficient, say so directly. Answer:""",
        )
        formatted_prompt = prompt_template.format(context=context, question=query)
        response = self.llm.invoke([HumanMessage(content=formatted_prompt)])
        return response.content

    def generate_response_simple(self, query: str, context: str) -> str:
        prompt = f"Based on this context: {context}\n\nQuestion: {query}\n\nAnswer:"
        response = self.llm.invoke([HumanMessage(content=prompt)])
        return response.content


In [53]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 236.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_76',
  'content': 'Page 4: Why RAG is useful\nRetrieval-Augmented Generation combines a language model with a searchable collection of\nexternal information. The language model provides reasoning and fluent answer generation, while\nthe retrieval system provides relevant source context. This separation is useful because source\ninformation can change independently from the model. It also allows an application to answer\nquestions about private documents or recent events that were not included in model training.\nIn practice, quality depends on accurate source preparation. Remove repeated headers, broken\ncharacters, and irrelevant boilerplate where possible. Retain meaningful headings and stable\nidentifiers because they help a retrieval system distinguish similar passages. Before deployment,\ntest representative questions and inspect both the retrieved chunks and the final answer.\nA retrieval pipeline should measure recall, precision, source coverage, and answer grounded

In [54]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.1,
)

def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=0.0)
    if not results:
        return "No relevant context found to answer the question."

    context = "\n\n".join(doc["content"] for doc in results)
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content


In [55]:
answer=rag_simple("What is attention mechanism?", rag_retriever, gemini_llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 103.24it/s]
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
The provided context does not contain information about the attention mechanism.


<br>

>### Enhanced rag pipeline

In [56]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([HumanMessage(content=prompt.format(context=context, query=query))])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is text chunk overlap?", rag_retriever, gemini_llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is text chunk overlap?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 98.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Chunk overlap copies some text between neighboring chunks, helping information that crosses a boundary remain available for retrieval.
Sources: [{'source': 'E:\\Learning\\Generative Ai\\Rag\\data\\pdf\\chunking_20.pdf', 'page': 19, 'score': 0.5008894205093384, 'preview': 'Page 20: Recursive text splitting\nRecursiveCharacterTextSplitter commonly attempts to split at paragraph boundaries, then sentence\nboundaries, and finally word or character boundaries. Paragraph-aware splitting usually produces\nmore readable chunks than a single fixed character split. Chunk overlap ...'}, {'source': 'chunking_20.pdf', 'page': 19, 'score': 0.5008894205093384, 'preview': 'Page 20: Recursive text splitting\nRecursiveCharacterTextSplitter commonly attempts to split at paragraph boundaries, then sentence\nboundaries, and finally word or character boundaries. Paragraph-aware splitting usually produces\nmore readable chunks than a single fixed character split. Chunk overlap ...'}, {'source': 'chun

In [59]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([HumanMessage(content=summary_prompt)])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, gemini_llm)
result = adv_rag.query("what are loaders used for?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what are loaders used for?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 126.97it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
LangChain Document Loaders and Data Sources
 Learning guide created for RAG document loader and retrieval practice.
Page 1: Purpose of a document loader
A document loader reads data from a source and converts it into LangChain Document objects.
Each Document normally contains page_content, which holds the extracted text, and meta

data,
which describes properties such as file name, source location, and page number. Preserving
metadata is important because it supports citations and can be used for filtering during retrieval.
In practice, quality depends on accurate source preparation. Remove repeated headers, broken
characters, and irrelevant boilerplate where possible. Retain meaningful headings and stable
identifiers because they help a retrieval system distinguish similar passages. Before deployment,
test representative questions and inspect both the retrieved chunks and the final answer.
A retrieval pipeline should measure recall, precision, source coverage, and answer groundedness.

LangChain Document Loaders and Data Sources
 Learning guide created for RAG document loader and retrieval practice.
Page 1: Purpose of a document loader
A document loader reads data from a source and converts it into LangChain Document objects.
Each Document normally contains page_content, which holds the extracted text, and meta